In [1]:
import json
from pathlib import Path
import google.generativeai as genai

e:\DS300-UIT-RecommenderSystem\DS300-venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load data

In [2]:
import pandas as pd
import ast
import re

# Load dữ liệu món ăn
FOOD_DATA_PATH = "../data/all_recipes_final.csv"
df_foods = pd.read_csv(FOOD_DATA_PATH)

print(f"Đã load {len(df_foods)} món ăn từ dataset\n")
print("Cấu trúc dữ liệu:")
print(df_foods.columns.tolist())
print(f"\nSố món theo loại:")
print(df_foods['type_of_food'].value_counts())

print("\nDataset đã sẵn sàng cho hệ thống gợi ý")

Đã load 10335 món ăn từ dataset

Cấu trúc dữ liệu:
['title', 'type_of_food', 'link', 'description', 'ingredients', 'step', 'note', 'num_of_ingredients', 'cook_time', 'num_of_people', 'calories', 'source']

Số món theo loại:
type_of_food
Món bánh                       2066
Món chiên                       860
Món xào                         848
Món kho                         580
Món canh                        574
Ăn vặt                          499
Món hấp                         477
Thức uống                       472
Món ngon hàng ngày              414
Món nướng                       386
Món nước                        344
Món gỏi - salad                 321
Món chính                       259
Món chay                        239
Món cháo                        229
Món chè                         210
Món tráng miệng                 169
Món lẩu                         164
Món khô - mắm                   158
Ngày lễ Tết                     117
Món ngon cho cuối tuần          112
Món kem

## 2. Model Config

In [3]:
# CONFIG
import os
from dotenv import load_dotenv
load_dotenv()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
genai.configure(api_key=GOOGLE_API_KEY)

MODEL = "models/gemini-2.5-flash-lite"

STATE_FILE = "dialog_state.json"

# INIT EMPTY STATE IF NOT EXIST
DEFAULT_STATE = {
    "hard_constraints": {
        "type_of_food": [],  # Món Tết, món chính, món phụ, etc.
        "ingredients": [],   # Thành phần chính (thịt, hải sản, rau, etc.)
        "cook_time": [],     # Thời gian nấu (nhanh <30p, trung bình 30-60p, lâu >60p)
        "algeric": []        # Dị ứng với thành phần nào không
    },
    "soft_constraints": {
        "num_of_people": [],  # Số người ăn
        "calories": [],       # Mức calories (thấp, trung bình, cao)
    },
    "recommended_items": [],
    "accepted_items": [],
    "rejected_items": []
}

In [4]:
def load_state():
    if Path(STATE_FILE).exists():
        return json.load(open(STATE_FILE, "r", encoding="utf-8"))
    else:
        json.dump(DEFAULT_STATE, open(STATE_FILE, "w", encoding="utf-8"), indent=4)
        return DEFAULT_STATE


def save_state(state):
    json.dump(state, open(STATE_FILE, "w", encoding="utf-8"), indent=4, ensure_ascii=False)

In [ ]:
# ============================================================
# 1. INTENT CLASSIFICATION (Provide Preference, Inquire, Accept, Reject)
# ============================================================

def classify_intent(user_prompt):
    prompt = f"""
            Phân loại Ý ĐỊNH của người dùng trong hệ thống gợi ý hội thoại.

            Các ý định có thể có (có thể nhiều ý định):
            - "Cung cấp sở thích" - người dùng nói những gì họ muốn (loại món, nguyên liệu, thời gian nấu, số người, calories, dịp đặc biệt)
            - "Hỏi thông tin" - người dùng đặt câu hỏi
            - "Chấp nhận gợi ý" - người dùng chấp nhận món được gợi ý
            - "Từ chối gợi ý" - người dùng từ chối món được gợi ý

            Ví dụ:
            - "Tôi muốn tìm món Tết" → ["Cung cấp sở thích"]
            - "Món nào nấu nhanh cho 2 người" → ["Cung cấp sở thích"]
            - "Bạn gợi ý món gì?" → ["Hỏi thông tin"]
            - "Món này hay đấy" → ["Chấp nhận gợi ý"]
            - "Không, tôi không thích món này" → ["Từ chối gợi ý"]
            - "Tôi thích món này" → ["Chấp nhận gợi ý"]
            - "Cho tôi món khác" → ["Từ chối gợi ý"]

            Người dùng nói: "{user_prompt}"

            Chỉ trả về mảng JSON các chuỗi ý định (ví dụ: ["Cung cấp sở thích"]).
            """
    resp = genai.GenerativeModel(MODEL).generate_content(prompt)

    try:
        text = resp.text.strip()
        # Remove markdown code blocks if present
        if "```" in text:
            # Extract content between ``` markers
            parts = text.split("```")
            if len(parts) >= 2:
                text = parts[1]
                # Remove language identifier if present
                if text.startswith("json"):
                    text = text[4:]
                text = text.strip()
        
        # Remove any leading/trailing whitespace and newlines
        text = text.strip()
        return json.loads(text)
    except Exception as e:
        print(f"Error parsing intent JSON: {e}")
        print(f"Response: {resp.text}")
        return []

In [ ]:
# ============================================================
# 2. STATE UPDATE USING LLM
# ============================================================

def update_state(user_prompt, intents, state):
    # ---- Update Preferences ----
    if "Cung cấp sở thích" in intents:
        # Check if user is providing complete info upfront or answering a specific question
        all_fields_empty = all(len(state["hard_constraints"][key]) == 0 for key in ["type_of_food", "ingredients", "cook_time", "algeric"])
        
        # Detect which field we're currently asking about
        current_asking_field = None
        for key in ["type_of_food", "ingredients", "cook_time", "algeric"]:
            if len(state["hard_constraints"][key]) == 0:
                current_asking_field = key
                break
        
        field_context = ""
        # Only restrict to single field if we're in the middle of asking step-by-step
        if current_asking_field and not all_fields_empty:
            field_names = {
                "type_of_food": "LOẠI MÓN (type_of_food)",
                "ingredients": "NGUYÊN LIỆU (ingredients)",
                "cook_time": "THỜI GIAN NẤU (cook_time)",
                "algeric": "DỊ ỨNG (algeric)"
            }
            field_context = f"\n**QUAN TRỌNG: Hiện tại đang hỏi về {field_names[current_asking_field]}. CHỈ được cập nhật trường này, GIỮ NGUYÊN tất cả các trường khác.**\n"
        elif all_fields_empty:
            field_context = f"\n**QUAN TRỌNG: Người dùng đang cung cấp thông tin đầy đủ ngay từ đầu. Hãy cập nhật TẤT CẢ các trường mà người dùng đã đề cập trong câu nói.**\n"
        
        prompt = f"""
                Bạn cập nhật trạng thái hội thoại JSON dựa trên sở thích của người dùng.

                Người dùng nói: "{user_prompt}"
                {field_context}
                JSON hiện tại:
                {json.dumps(state, indent=4, ensure_ascii=False)}

                Quy tắc:
                - **Nếu người dùng cung cấp đầy đủ thông tin trong một câu**: Cập nhật TẤT CẢ các trường được đề cập
                - **Nếu đang hỏi từng bước**: CHỈ cập nhật trường đang được hỏi, GIỮ NGUYÊN các trường khác
                - **Phân tích kỹ câu người dùng**: Xác định người dùng đề cập đến những trường nào

                **CHÚ Ý: Phân tích kỹ câu người dùng để xác định TẤT CẢ các trường được đề cập!**

                - **type_of_food**: 
                  + ✅ Cập nhật nếu người dùng đề cập loại món: "món Tết", "món chính", "món phụ" 
                    → Ví dụ: "món Tết" → type_of_food = ["món tết"]
                  + ❌ Nếu KHÔNG đề cập → GIỮ NGUYÊN []

                - **ingredients**: 
                  + ✅ Cập nhật nếu người dùng đề cập nguyên liệu
                    - Nêu cụ thể: "nem chua", "thịt lợn" → ingredients = ["nem chua"]
                    - "Gì cũng được", "bất kỳ" → ingredients = ["everything"]
                  + ❌ Nếu KHÔNG đề cập → GIỮ NGUYÊN []
                  
                - **cook_time**: ⚠️ QUAN TRỌNG
                  + ✅ Cập nhật nếu người dùng đề cập thời gian (kể cả nói KHÔNG quan tâm)
                    - "Không yêu cầu", "bất kỳ", "không quan trọng", "gì cũng được" 
                      → cook_time = ["nhanh", "trung bình", "lâu"]
                    - Nêu cụ thể: "nhanh" → ["nhanh"], "lâu" → ["lâu"]
                  + ❌ Nếu KHÔNG đề cập gì về thời gian → GIỮ NGUYÊN []
                  
                - **algeric**: ⚠️ QUAN TRỌNG
                  + ✅ Cập nhật nếu người dùng đề cập dị ứng (kể cả nói KHÔNG dị ứng)
                    - "Không dị ứng", "không bị dị ứng", "tôi không dị ứng" 
                      → algeric = ["none"]
                    - Nêu cụ thể: "dị ứng tôm" → algeric = ["tôm"]
                  + ❌ Nếu KHÔNG đề cập gì về dị ứng → GIỮ NGUYÊN []

                - **num_of_people**: ⚠️ QUAN TRỌNG (soft_constraints)
                  + ✅ Cập nhật nếu người dùng đề cập số người (kể cả nói KHÔNG quan tâm)
                    - "Không cần quan tâm số người", "không quan trọng" 
                      → num_of_people = ["none"]
                    - Nêu cụ thể: "4 người" → num_of_people = ["4"]
                  + ❌ Nếu KHÔNG đề cập gì về số người → GIỮ NGUYÊN []

                - **calories**: ⚠️ QUAN TRỌNG (soft_constraints)
                  + ✅ Cập nhật nếu người dùng đề cập calories/kcal (kể cả nói KHÔNG quan tâm)
                    - "Không cần quan tâm kcal", "không quan tâm calories" 
                      → calories = ["none"]
                    - Nêu cụ thể: "ít calo" → calories = ["thấp"]
                  + ❌ Nếu KHÔNG đề cập gì về calories → GIỮ NGUYÊN []

                **Ví dụ MẪU - HÃY HỌC KỸ:**

                Câu 1: "Gợi ý món Tết, nguyên liệu nem chua, không yêu cầu thời gian nấu, tôi không bị dị ứng, không cần quan tâm số người ăn và số kcal"

                Phân tích từng phần:
                - Đề cập "món Tết" → Cập nhật type_of_food
                - Đề cập "nem chua" → Cập nhật ingredients  
                - Đề cập "không yêu cầu thời gian" → Cập nhật cook_time (tất cả 3 loại)
                - Đề cập "không bị dị ứng" → Cập nhật algeric = none
                - Đề cập "không cần quan tâm số người" → Cập nhật num_of_people = none
                - Đề cập "không cần quan tâm kcal" → Cập nhật calories = none

                → Tất cả 6 trường phải được cập nhật!

                QUAN TRỌNG: Chỉ trả về JSON thuần túy, KHÔNG có markdown, KHÔNG có ```json, KHÔNG có giải thích.
                Trả về đúng format JSON hợp lệ với tất cả dấu phay và dấu ngoặc đúng vị trí.
                """
        resp = genai.GenerativeModel(MODEL).generate_content(prompt)

        try:
            # Remove markdown code blocks if present
            text = resp.text.strip()
            if "```" in text:
                # Extract content between ``` markers
                parts = text.split("```")
                if len(parts) >= 2:
                    text = parts[1]
                    # Remove language identifier if present
                    if text.startswith("json"):
                        text = text[4:]
                    elif text.startswith("JSON"):
                        text = text[4:]
                    text = text.strip()
            # Remove any leading/trailing whitespace
            text = text.strip()
            
            new_state = json.loads(text)
            
            # Validate that new_state has required structure
            if "hard_constraints" in new_state and "soft_constraints" in new_state:
                state = new_state
            else:
                print("⚠️ LLM trả về JSON không đầy đủ cấu trúc, giữ nguyên state cũ")
                print(f"Response: {text[:200]}...")
        except json.JSONDecodeError as e:
            print(f"❌ Error parsing JSON: {e}")
            print(f"Response text: {resp.text[:300]}...")
            print("⚠️ Giữ nguyên state cũ, tiếp tục hỏi...")
            # Không cập nhật state, giữ nguyên
        except Exception as e:
            print(f"❌ Unexpected error: {e}")
            print(f"Response text: {resp.text[:300]}...")
            print("⚠️ Giữ nguyên state cũ, tiếp tục hỏi...")

    if "Chấp nhận gợi ý" in intents:
        # Lấy món từ recommended_items (món cuối cùng được gợi ý)
        if state["recommended_items"]:
            item = state["recommended_items"][-1]
            if item not in state["accepted_items"]:
                state["accepted_items"].append(item)

    if "Từ chối gợi ý" in intents:
        # Lấy món từ recommended_items (món cuối cùng được gợi ý)
        if state["recommended_items"]:
            item = state["recommended_items"][-1]
            if item not in state["rejected_items"]:
                state["rejected_items"].append(item)
    
    return state

In [7]:
# ============================================================
# 3. ACTION SELECTION
# ============================================================

def select_action(intents, state):
    # If user asks a question:
    if "Hỏi thông tin" in intents:
        return "Answer"

    # Check all hard_constraints for missing values
    for key in state["hard_constraints"]:
        if len(state["hard_constraints"][key]) == 0:
            return "Request Hard Constraint"
    
    # All hard constraints filled, check soft constraints
    all_soft_empty = all(len(state["soft_constraints"][key]) == 0 for key in state["soft_constraints"])
    
    if all_soft_empty:
        return "Request Soft Constraint"
    
    # Everything filled → recommend
    return "Recommend and Explain"

## Hàm lọc và gợi ý món ăn dựa trên constraints

In [8]:
def filter_dishes_by_constraints(state, df_foods):
    """
    Lọc món ăn dựa trên hard constraints và soft constraints
    Returns: DataFrame các món ăn phù hợp
    """
    filtered_df = df_foods.copy()
    
    # 1. HARD CONSTRAINTS - bắt buộc phải match
    
    # Type of food - CASE INSENSITIVE matching
    if state["hard_constraints"]["type_of_food"]:
        type_list = state["hard_constraints"]["type_of_food"]
        # Convert to lowercase for comparison
        type_list_lower = [t.lower() for t in type_list]
        filtered_df = filtered_df[filtered_df['type_of_food'].str.lower().isin(type_list_lower)]
    
    # Ingredients - check if món có chứa nguyên liệu user muốn
    if state["hard_constraints"]["ingredients"]:
        ingredient_keywords = state["hard_constraints"]["ingredients"]
        # Nếu là "everything" thì không lọc gì cả (user ăn gì cũng được)
        if "everything" not in [item.lower() for item in ingredient_keywords]:
            # Chỉ lọc khi có nguyên liệu cụ thể
            def has_ingredient(row):
                ingredients_str = str(row['ingredients']).lower()
                title_str = str(row['title']).lower()
                # Tìm kiếm trong cả ingredients và title
                full_text = ingredients_str + ' ' + title_str
                # Check if any keyword appears in the full text
                for keyword in ingredient_keywords:
                    keyword_lower = keyword.lower()
                    if keyword_lower in full_text:
                        return True
                return False
            filtered_df = filtered_df[filtered_df.apply(has_ingredient, axis=1)]
    
    # Allergic - loại bỏ món có thành phần user dị ứng
    if state["hard_constraints"]["algeric"]:
        allergic_items = state["hard_constraints"]["algeric"]
        # Nếu là "none" thì không lọc gì cả (user không dị ứng)
        if "none" not in [item.lower() for item in allergic_items]:
            # Chỉ lọc khi có thành phần dị ứng thực sự
            def is_safe(row):
                ingredients_str = str(row['ingredients']).lower()
                title_str = str(row['title']).lower()
                # Kiểm tra trong cả ingredients và title
                full_text = ingredients_str + ' ' + title_str
                return not any(allergen.lower() in full_text for allergen in allergic_items)
            filtered_df = filtered_df[filtered_df.apply(is_safe, axis=1)]
    
    # Cook time
    if state["hard_constraints"]["cook_time"]:
        cook_time_prefs = [item.lower() for item in state["hard_constraints"]["cook_time"]]
        
        # Nếu có cả 3 loại hoặc rỗng thì không lọc (chấp nhận tất cả)
        if not (len(cook_time_prefs) == 3 and "nhanh" in cook_time_prefs and "trung bình" in cook_time_prefs and "lâu" in cook_time_prefs):
            def match_cook_time(row):
                cook_time_str = str(row['cook_time']).lower()
                try:
                    time_match = re.search(r'(\d+)', cook_time_str)
                    if time_match:
                        minutes = int(time_match.group(1))
                        # Check if any preference matches
                        for pref in cook_time_prefs:
                            if "nhanh" in pref or "quick" in pref:
                                if minutes <= 30:
                                    return True
                            elif "trung bình" in pref or "medium" in pref:
                                if 30 < minutes <= 60:
                                    return True
                            elif "lâu" in pref or "long" in pref or "chậm" in pref:
                                if minutes > 60:
                                    return True
                except:
                    pass
                return False
            filtered_df = filtered_df[filtered_df.apply(match_cook_time, axis=1)]
    
    # 2. SOFT CONSTRAINTS - ưu tiên nhưng không bắt buộc
    # Sắp xếp theo độ phù hợp với soft constraints
    
    # Num of people
    if state["soft_constraints"]["num_of_people"]:
        target_people = state["soft_constraints"]["num_of_people"][0]
        try:
            target_num = int(re.search(r'(\d+)', str(target_people)).group(1))
            filtered_df['people_score'] = filtered_df['num_of_people'].apply(
                lambda x: abs(int(re.search(r'(\d+)', str(x)).group(1)) - target_num) if pd.notna(x) and re.search(r'(\d+)', str(x)) else 999
            )
        except:
            filtered_df['people_score'] = 0
    else:
        filtered_df['people_score'] = 0
    
    # Calories
    if state["soft_constraints"]["calories"]:
        calorie_pref = str(state["soft_constraints"]["calories"][0]).lower()
        def match_calories(row):
            cal = row['calories']
            if pd.notna(cal):
                try:
                    cal_num = float(cal)
                    if "thấp" in calorie_pref or "low" in calorie_pref:
                        return cal_num <= 500
                    elif "cao" in calorie_pref or "high" in calorie_pref:
                        return cal_num >= 1000
                    else:  # trung bình
                        return 500 < cal_num < 1000
                except (ValueError, TypeError):
                    pass
            return True
        filtered_df['calorie_match'] = filtered_df.apply(match_calories, axis=1)
        filtered_df = filtered_df.sort_values('calorie_match', ascending=False)
    
    # Sort by soft constraint scores
    if 'people_score' in filtered_df.columns:
        filtered_df = filtered_df.sort_values('people_score')
    
    return filtered_df

def recommend_dish(state):
    """
    Gợi ý món ăn phù hợp nhất từ dataset dựa trên state
    Loại trừ các món đã bị từ chối
    """
    filtered = filter_dishes_by_constraints(state, df_foods)
    
    # Loại bỏ các món đã bị từ chối
    if state["rejected_items"]:
        filtered = filtered[~filtered['title'].isin(state["rejected_items"])]
    # Loại bỏ các món đã bị từ chối
    if len(filtered) == 0:
        return None, "Xin lỗi, không tìm thấy món ăn phù hợp với yêu cầu của bạn. Bạn có thể thử nới lỏng điều kiện không?"
    
    # Lấy món đầu tiên (phù hợp nhất)
    top_dish = filtered.iloc[0]
    
    # Tạo explanation
    explanation = f"""
🍽️ **{top_dish['title']}**

📋 **Thông tin món ăn:**
- Loại món: {top_dish['type_of_food']}
- Thời gian nấu: {top_dish['cook_time']}
- Số người ăn: {top_dish['num_of_people']}
- Calories: {top_dish['calories']} kcal
- Số nguyên liệu: {top_dish['num_of_ingredients']}

📝 **Mô tả:** {top_dish['description']}

✅ **Lý do phù hợp:**
"""
    
    # Thêm lý do dựa trên constraints
    reasons = []
    if state["hard_constraints"]["type_of_food"]:
        reasons.append(f"- Đúng loại món bạn tìm: {', '.join(state['hard_constraints']['type_of_food'])}")
    if state["hard_constraints"]["cook_time"]:
        reasons.append(f"- Thời gian nấu như mong muốn")
    
    explanation += "\n".join(reasons)
    explanation += "\n\n❓ **Bạn có muốn làm món này không?** (trả lời: có/ok/được hoặc không/khác)"
    
    return top_dish, explanation

def get_recipe_details(dish_title):
    """
    Lấy chi tiết công thức nấu ăn (nguyên liệu và các bước)
    """
    dish_row = df_foods[df_foods['title'] == dish_title]
    
    if len(dish_row) == 0:
        return "Không tìm thấy thông tin món ăn."
    
    dish = dish_row.iloc[0]
    
    # Parse ingredients
    try:
        if isinstance(dish['ingredients'], str):
            ingredients_list = ast.literal_eval(dish['ingredients'])
        else:
            ingredients_list = dish['ingredients']
    except:
        ingredients_list = [dish['ingredients']]
    
    # Parse steps
    try:
        if isinstance(dish['step'], str):
            steps_list = ast.literal_eval(dish['step'])
        else:
            steps_list = dish['step']
    except:
        steps_list = [dish['step']]
    
    # Format output
    recipe = f"""
✨ **CÔNG THỨC NẤU: {dish['title']}**

🛒 **NGUYÊN LIỆU CẦN CHUẨN BỊ:**
"""
    
    for i, ingredient in enumerate(ingredients_list, 1):
        recipe += f"{i}. {ingredient}\n"
    
    recipe += f"\n👨‍🍳 **CÁC BƯỚC THỰC HIỆN:**\n"
    
    for i, step in enumerate(steps_list, 1):
        recipe += f"\n**Bước {i}:**\n{step}\n"
    
    if pd.notna(dish['note']) and dish['note']:
        recipe += f"\n💡 **GHI CHÚ:** {dish['note']}"
    
    recipe += f"\n\n✅ Chúc bạn nấu ăn ngon miệng! 🍽️"
    
    return recipe


print("✅ Đã tạo hàm filter và recommend từ dataset thực tế")

✅ Đã tạo hàm filter và recommend từ dataset thực tế


In [9]:
# ============================================================
# 4. GENERATE RESPONSE (Answer / Request Info / Recommend)
# ============================================================

def generate_bot_response(user_prompt, action, state):
    if action == "Request Hard Constraint":
        # Tìm hard constraint còn thiếu và hỏi bằng tiếng Việt
        for key in state["hard_constraints"]:
            if len(state["hard_constraints"][key]) == 0:
                questions = {
                    "type_of_food": "Bạn muốn tìm loại món gì? (ví dụ: Món Tết, món chính, món phụ...)",
                    "ingredients": "Bạn muốn món có nguyên liệu gì? (ví dụ: thịt lợn, hải sản, rau... hoặc 'gì cũng được')",
                    "cook_time": "Bạn muốn món nấu nhanh hay chậm? (nhanh: <30p, trung bình: 30-60p, lâu: >60p)",
                    "algeric": "Bạn có dị ứng với thành phần nào không? (nếu không thì nói 'không')"
                }
                return questions.get(key, f"Bạn có yêu cầu gì về {key}?")
        return "Tôi cần thêm thông tin để giúp bạn."
    
    elif action == "Request Soft Constraint":
        # Hỏi về soft constraints
        return "Bạn có muốn cho biết số người ăn hoặc mức calories không? (hoặc nói 'không' để bỏ qua)"

    elif action == "Answer":
        prompt = f"""
Người dùng hỏi: "{user_prompt}"

Trạng thái:
{json.dumps(state, indent=4, ensure_ascii=False)}

Tạo câu trả lời thông tin bằng tiếng Việt dựa trên sở thích và ngữ cảnh.
Trả lời ngắn gọn, thân thiện và hữu ích.
"""
        return genai.GenerativeModel(MODEL).generate_content(prompt).text

    elif action == "Recommend and Explain":
        # Sử dụng hàm recommend_dish để lấy món ăn thực từ dataset
        dish, explanation = recommend_dish(state)
        
        if dish is None:
            return explanation  # Trả về thông báo không tìm thấy
        
        # Lưu món đã gợi ý vào recommended_items
        if dish['title'] not in state["recommended_items"]:
            state["recommended_items"].append(dish['title'])
            save_state(state)
        
        return explanation
    

    return "Tôi không hiểu yêu cầu của bạn. Bạn có thể nói rõ hơn không?"

In [ ]:
print("=== Hệ thống Gợi ý Món Ăn (Food Recommender) ===\n")

# Reset state to empty at the beginning of each run
with open(STATE_FILE, "w", encoding="utf-8") as f:
    json.dump(DEFAULT_STATE, f, indent=4, ensure_ascii=False)
print("State reset to default.\n")

# Initial greeting from bot
print("BOT: Xin chào! Tôi là trợ lý gợi ý món ăn. Bạn muốn tìm món ăn gì? (ví dụ: món Tết, món nhanh, món cho 4 người...)\n")

while True:
    user_prompt = input("USER: ")

    if user_prompt.lower() == "exit" or user_prompt.lower() == "stop":
        break

    state = load_state()

    # Step 1: Classify intent
    intents = classify_intent(user_prompt)
    
    # Force "Cung cấp sở thích" if we're in the middle of asking hard constraints
    # Check if any hard constraint is empty (we're still collecting info)
    asking_hard_constraints = any(len(state["hard_constraints"][key]) == 0 for key in state["hard_constraints"])
    if asking_hard_constraints and "Cung cấp sở thích" not in intents:
        # User is responding to our question, force it as providing preference
        intents = ["Cung cấp sở thích"]
    
    # Special handling: If user says "không" for algeric, force update
    if any(word in user_prompt.lower() for word in ["không", "khong", "không có", "không dị ứng"]):
        if state["hard_constraints"]["algeric"] == []:
            # Directly update algeric to "none" to avoid infinite loop
            state["hard_constraints"]["algeric"] = ["none"]
            save_state(state)
            # Force classify as providing preference
            if "Cung cấp sở thích" not in intents:
                intents.append("Cung cấp sở thích")
    
    # Check if user rejects soft constraint (says "không", "no", "skip", etc.)
    if any(word in user_prompt.lower() for word in ["không", "khong", "no", "skip", "nope", "don't", "dont", "bỏ qua"]):
        # Check if all hard constraints are filled
        all_hard_filled = all(len(state["hard_constraints"][key]) > 0 for key in state["hard_constraints"])
        if all_hard_filled:
            # User declined soft constraints, set to "none" directly
            if len(state["soft_constraints"]["num_of_people"]) == 0:
                state["soft_constraints"]["num_of_people"] = ["none"]
            if len(state["soft_constraints"]["calories"]) == 0:
                state["soft_constraints"]["calories"] = ["none"]
            save_state(state)
            
            # Now go to recommend
            action = "Recommend and Explain"
            reply = generate_bot_response(user_prompt, action, state)
            print(f"BOT: {reply}\n")
            continue

    # Step 2: Update state
    state = update_state(user_prompt, intents, state)
    save_state(state)

    # Step 3: Select action
    action = select_action(intents, state)

    # Step 4: Generate response
    reply = generate_bot_response(user_prompt, action, state)
    print(f"BOT: {reply}\n")
    
    # Step 5: Check if user accepted recommendation
    if "Chấp nhận gợi ý" in intents and state["accepted_items"]:
        # Show recipe details
        accepted_dish = state["accepted_items"][-1]
        recipe = get_recipe_details(accepted_dish)
        print(f"\n{recipe}\n")

=== Hệ thống Gợi ý Món Ăn (Food Recommender) ===

State reset to default.

BOT: Xin chào! Tôi là trợ lý gợi ý món ăn. Bạn muốn tìm món ăn gì? (ví dụ: món Tết, món nhanh, món cho 4 người...)

BOT: Bạn muốn tìm loại món gì? (ví dụ: Món Tết, món chính, món phụ...)

BOT: Bạn muốn món nấu nhanh hay chậm? (nhanh: <30p, trung bình: 30-60p, lâu: >60p)

BOT: Bạn có dị ứng với thành phần nào không? (nếu không thì nói 'không')

BOT: Xin lỗi, không tìm thấy món ăn phù hợp với yêu cầu của bạn. Bạn có thể thử nới lỏng điều kiện không?

BOT: Xin lỗi, không tìm thấy món ăn phù hợp với yêu cầu của bạn. Bạn có thể thử nới lỏng điều kiện không?

